load the included entries and frozen source records

In [1]:
import csv
import hashlib
import io
import json
import math
import re
from datetime import date
from html import escape
from pathlib import Path

import gemmi
from IPython.display import HTML, display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
data_root = project_root / "data"
manifest_path = data_root / "fibril_manifest.json"
review_path = data_root / "fibril_inclusion_review.csv"
manifest = json.loads(manifest_path.read_text())
with review_path.open(newline="") as source:
    review = list(csv.DictReader(source))
included_entries = sorted(
    (row for row in review if row["decision"] == "include"), key=lambda row: row["pdb_id"]
)
assert included_entries, "no entries approved for curation"
assert len({row["pdb_id"] for row in included_entries}) == len(included_entries)
atlas_entries = {entry["pdb_id"]: entry for entry in manifest["atlas_entries"]}
raw_root = data_root / "raw" / "alpha_synuclein_fibrils"

retain raw values and source locations for each metadata field

In [2]:
def read_fibril_source(path):
    content = path.read_bytes()
    relative_path = path.relative_to(project_root).as_posix()
    assert hashlib.sha256(content).hexdigest() == manifest["files"][relative_path]["sha256"], relative_path
    return content


def cif_category_rows(block, category):
    columns = block.get_mmcif_category(category)
    return [dict(zip(columns, values)) for values in zip(*columns.values())] if columns else []


def record_fibril_field(row, field, value, source, raw=None, reason="not reported in the downloaded structured records"):
    missing = value is None or value is False or value in ("", [], {})
    row[field] = "unknown" if missing else value if isinstance(value, (str, int, float)) else json.dumps(value, sort_keys=True)
    row[field + "_raw"] = json.dumps(value if raw is None else raw, sort_keys=True)
    row[field + "_source"] = source
    row[field + "_unknown_reason"] = reason if missing else ""


def single_reported_value(values):
    reported = [value for value in values if value is not None]
    return reported[0] if reported and all(value == reported[0] for value in reported) else None


paper_review_fields = {
    "source_preparation_id": "shared sample or preparation identity requires publication review",
    "clinical_source": "patient origin requires publication review; natural and recombinant describe deposited material",
    "seeding_status": "seeding history requires publication review",
    "construct_boundaries": "true construct boundaries require publication review; deposited sequences can describe only the model",
    "growth_ph": "specimen buffer ph does not establish fibril growth ph",
    "growth_buffer_salt": "growth buffer and salt require publication review; specimen components are retained separately",
    "growth_temperature_k": "growth temperature requires publication review; vitrification temperature is retained separately",
    "research_group": "laboratory identity requires publication review; authors are retained separately",
    "protofilament_count": "protofilament count requires publication review; chain counts are not equivalent",
}

extract deposited annotations and specimen conditions without inferring clinical origin or growth conditions

In [3]:
metadata_rows = []
for approved in included_entries:
    pdb_id = approved["pdb_id"]
    coordinate_path = raw_root / "coordinates" / f"{pdb_id}.cif"
    entry_path = raw_root / "pdb_metadata" / f"{pdb_id}.json"
    block = gemmi.cif.read_string(read_fibril_source(coordinate_path).decode()).sole_block()
    entry = json.loads(read_fibril_source(entry_path))
    assert block.find_value("_entry.id").upper() == entry["rcsb_id"] == pdb_id
    cif_source = coordinate_path.relative_to(project_root).as_posix()
    pdb_source = entry_path.relative_to(project_root).as_posix()
    atlas_source = f"data/raw/alpha_synuclein_fibrils/amyloid_atlas.html::pdb_id={pdb_id}/Fibril Origins"
    entities = [
        entity for entity in cif_category_rows(block, "_entity.")
        if re.search(r"(alpha|α)[ -]?synuclein", entity.get("pdbx_description") or "", re.IGNORECASE)
    ]
    alpha_ids = {entity["id"] for entity in entities}
    assert alpha_ids, pdb_id
    polymers = [polymer for polymer in cif_category_rows(block, "_entity_poly.") if polymer["entity_id"] in alpha_ids]
    raw_sequences = {polymer["entity_id"]: polymer.get("pdbx_seq_one_letter_code_can") for polymer in polymers}
    sequences = {entity_id: "".join(sequence.split()) for entity_id, sequence in raw_sequences.items() if sequence}
    raw_monomers = {polymer["entity_id"]: polymer.get("pdbx_seq_one_letter_code") for polymer in polymers}
    monomers = {
        entity_id: sorted(set(re.findall(r"\(([^)]+)\)", sequence)))
        for entity_id, sequence in raw_monomers.items() if sequence and "(" in sequence
    }
    origins = sorted({
        assembly["source"] for assembly in entry.get("em_entity_assembly", [])
        if alpha_ids.intersection(assembly.get("entity_id_list") or []) and assembly.get("source")
    })
    row = {
        "pdb_id": pdb_id,
        "coordinate_path": coordinate_path.relative_to(project_root).as_posix(),
        "source_dataset_version": manifest["dataset_version"],
    }
    record_fibril_field(row, "study_doi", approved["publication_url"].split("doi.org/", 1)[-1],
                        f"data/fibril_inclusion_review.csv::pdb_id={pdb_id}/publication_url", raw=approved["publication_url"])
    record_fibril_field(row, "species", approved["species"], f"data/fibril_inclusion_review.csv::pdb_id={pdb_id}/species")
    record_fibril_field(row, "deposited_source", origins, pdb_source + "::em_entity_assembly[*].source")
    record_fibril_field(row, "source_notes", atlas_entries[pdb_id]["origins_text"], atlas_source)
    record_fibril_field(row, "deposited_sequences", sequences, cif_source + "::_entity_poly.pdbx_seq_one_letter_code_can", raw=raw_sequences)
    record_fibril_field(row, "fragment_annotations", [entity["pdbx_fragment"] for entity in entities if entity.get("pdbx_fragment")],
                        cif_source + "::_entity.pdbx_fragment")
    record_fibril_field(row, "mutation_annotations", [entity["pdbx_mutation"] for entity in entities if entity.get("pdbx_mutation")],
                        cif_source + "::_entity.pdbx_mutation", reason="no mutation annotation deposited; this does not establish wild type")
    record_fibril_field(row, "nonstandard_monomers", monomers, cif_source + "::_entity_poly.pdbx_seq_one_letter_code",
                        raw=raw_monomers, reason="no nonstandard monomer annotation deposited; this does not establish unmodified protein")
    citation = entry.get("rcsb_primary_citation", {})
    record_fibril_field(row, "reported_authors", citation.get("rcsb_authors"), pdb_source + "::rcsb_primary_citation.rcsb_authors")
    release = entry.get("rcsb_accession_info", {}).get("initial_release_date")
    record_fibril_field(row, "release_date", release.split("T")[0] if release else None,
                        pdb_source + "::rcsb_accession_info.initial_release_date", raw=release)
    resolutions = entry.get("rcsb_entry_info", {}).get("resolution_combined", [])
    record_fibril_field(row, "resolution_angstrom", single_reported_value(resolutions),
                        pdb_source + "::rcsb_entry_info.resolution_combined", raw=resolutions,
                        reason="resolution missing or multiple differing values reported")
    buffers = []
    buffer_sources = []
    for emdb_id in entry.get("rcsb_entry_container_identifiers", {}).get("emdb_ids", []):
        emdb_path = raw_root / "emdb_metadata" / f"{emdb_id}.json"
        emdb = json.loads(read_fibril_source(emdb_path))
        assert emdb["emdb_id"] == emdb_id
        for determination_index, determination in enumerate(emdb.get("structure_determination_list", {}).get("structure_determination", [])):
            for preparation_index, preparation in enumerate(determination.get("specimen_preparation_list", {}).get("specimen_preparation", [])):
                buffers.append(preparation.get("buffer", {}))
                buffer_sources.append(
                    f"{emdb_path.relative_to(project_root).as_posix()}::structure_determination_list.structure_determination[{determination_index}]"
                    f".specimen_preparation_list.specimen_preparation[{preparation_index}].buffer"
                )
    ph_values = [buffer.get("ph") for buffer in buffers]
    record_fibril_field(row, "specimen_ph", single_reported_value(ph_values), json.dumps(buffer_sources), raw=ph_values,
                        reason="specimen ph missing or multiple differing values reported")
    components = [component for buffer in buffers for component in buffer.get("component", [])]
    record_fibril_field(row, "specimen_buffer_components", components, json.dumps(buffer_sources),
                        raw=[buffer.get("component") for buffer in buffers])
    temperatures = [preparation.get("chamber_temperature") for preparation in entry.get("em_vitrification", [])]
    record_fibril_field(row, "vitrification_temperature_k", single_reported_value(temperatures),
                        pdb_source + "::em_vitrification[*].chamber_temperature", raw=temperatures,
                        reason="pre-vitrification temperature missing or multiple differing values reported")
    for field, reason in paper_review_fields.items():
        record_fibril_field(row, field, None, atlas_source, raw=atlas_entries[pdb_id]["origins_text"], reason=reason)
    metadata_rows.append(row)

apply the first publication review to 6cu7 and 6cu8, retaining unresolved values for spot-checking

In [4]:
li_2018_sources = {
    "main": {
        "path": "data/raw/publications/li_2018/li_2018_main.pdf",
        "url": "https://www.nature.com/articles/s41467-018-05971-2.pdf",
        "sha256": "db06711ef6af619c7302c8c898374fb34ba3b1ce066f3c6b4a86c17a27203a8a",
        "retrieved_at": "2026-11-13T01:55:28.081853+00:00"
    },
    "supplement": {
        "path": "data/raw/publications/li_2018/li_2018_supplement.pdf",
        "url": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-018-05971-2/MediaObjects/41467_2018_5971_MOESM1_ESM.pdf",
        "sha256": "cfb3de3d0464fd8e04d0ae7156f626d7179349578cb0a58b20056d59a61dead8",
        "retrieved_at": "2026-11-13T01:55:27.341992+00:00"
    }
}
for paper_source in li_2018_sources.values():
    content = (project_root / paper_source["path"]).read_bytes()
    assert hashlib.sha256(content).hexdigest() == paper_source["sha256"], paper_source["path"]

li_2018_pdb_ids = {"6CU7", "6CU8"}
assert li_2018_pdb_ids <= {row["pdb_id"] for row in metadata_rows}
li_2018_values = {
    "clinical_source": (
        "in_vitro", "recombinant human aSyn", "main", "pp.1-2, abstract and results",
    ),
    "construct_boundaries": (
        "human alpha-synuclein 1-140; full-length", "1–140", "main",
        "p.2, results; p.7, expression and purification",
    ),
    "growth_buffer_salt": (
        "15 mM tetrabutylphosphonium bromide; base buffer unspecified",
        "15mM tetrabutylphosphonium bromide", "supplement", "p.3, supplementary figure 2 caption",
    ),
    "growth_temperature_k": (
        None, "room temperature", "supplement", "p.3, supplementary figure 2 caption",
    ),
    "growth_ph": (
        None, None, "main", "p.8, fibril preparation; selected condition",
    ),
    "seeding_status": (
        None, None, "main", "p.8, fibril preparation; structural sample",
    ),
    "research_group": (
        "UCLA collaboration: Jiang, Zhou, Eisenberg",
        {"affiliation": "UCLA", "supervision": "L.J., Z.H.Z., and D.S.E."},
        "main", "p.1, affiliations; p.9, author contributions",
    ),
    "protofilament_count": (
        2, "two intertwined protofilaments", "main", "p.2, cryo-em structures of two polymorphs",
    ),
    "source_preparation_id": (
        "li_2018_cryoem_preparation", "one preparation", "main",
        "p.2, introduction; p.9, data availability linking both pdb entries",
    ),
}
li_2018_unknown_reasons = {
    "growth_temperature_k": "room temperature reported without a numeric value; no kelvin value inferred",
    "growth_ph": "selected-preparation ph not stated; screening and specimen ph values are not assigned to growth",
    "seeding_status": "seed addition not explicitly stated for the structural preparation; later seeding assays are downstream experiments",
}
paper_curation_changes = []
for row in metadata_rows:
    if row["pdb_id"] not in li_2018_pdb_ids:
        continue
    assert row["study_doi"] == "10.1038/s41467-018-05971-2"
    for field, (value, raw, document, location) in li_2018_values.items():
        field_keys = [field, field + "_raw", field + "_source", field + "_unknown_reason"]
        previous = {key: row[key] for key in field_keys}
        source = f"{li_2018_sources[document]['path']}::{location}; doi:10.1038/s41467-018-05971-2"
        record_fibril_field(row, field, value, source, raw=raw, reason=li_2018_unknown_reasons.get(field, ""))
        paper_curation_changes.append({
            "pdb_id": row["pdb_id"],
            "field": field,
            "previous": previous,
            "current": {key: row[key] for key in field_keys},
        })

li_2018_review = {
    "study_doi": "10.1038/s41467-018-05971-2",
    "pdb_ids": sorted(li_2018_pdb_ids),
    "curated_by": "assistant",
    "curated_on": "2026-11-13",
    "review_status": "awaiting user spot-check",
    "group_label_note": "preparation and collaboration labels are curator-assigned from the paper, not author-issued identifiers",
    "sources": li_2018_sources,
    "changes": paper_curation_changes,
}

check the table and summarise missingness and distinct reported values

In [5]:
assert {row["pdb_id"] for row in metadata_rows} == {row["pdb_id"] for row in included_entries}
metadata_fields = [field for field in metadata_rows[0] if field + "_source" in metadata_rows[0]]
for row in metadata_rows:
    for field in metadata_fields:
        assert row[field + "_source"]
        assert bool(row[field + "_unknown_reason"]) == (row[field] == "unknown")
    if row["release_date"] != "unknown":
        date.fromisoformat(row["release_date"])
    for field in ("resolution_angstrom", "vitrification_temperature_k", "growth_temperature_k"):
        if row[field] != "unknown":
            assert math.isfinite(float(row[field])) and float(row[field]) > 0, (row["pdb_id"], field)
    if row["protofilament_count"] != "unknown":
        assert isinstance(row["protofilament_count"], int) and row["protofilament_count"] > 0, row["pdb_id"]
    if row["specimen_ph"] != "unknown":
        assert math.isfinite(float(row["specimen_ph"])) and 0 <= float(row["specimen_ph"]) <= 14, row["pdb_id"]

missingness = [
    {
        "field": field,
        "reported": sum(row[field] != "unknown" for row in metadata_rows),
        "unknown": sum(row[field] == "unknown" for row in metadata_rows),
        "distinct_reported": len({str(row[field]) for row in metadata_rows if row[field] != "unknown"}),
    }
    for field in metadata_fields
]

save an immutable table version with its input and code fingerprints

In [6]:
table_buffer = io.StringIO(newline="")
writer = csv.DictWriter(table_buffer, fieldnames=list(metadata_rows[0]), lineterminator="\n")
writer.writeheader()
writer.writerows(metadata_rows)
table_bytes = table_buffer.getvalue().encode()
table_sha256 = hashlib.sha256(table_bytes).hexdigest()
curation_root = data_root / "curated"
curation_root.mkdir(exist_ok=True)

notebook_path = project_root / "notebooks" / "curate_fibril_metadata.ipynb"
notebook = json.loads(notebook_path.read_text())
code_source = "\n\n".join("".join(cell["source"]) for cell in notebook["cells"] if cell["cell_type"] == "code")
curation_record = {
    "table_sha256": table_sha256,
    "source_dataset_version": manifest["dataset_version"],
    "source_manifest_sha256": hashlib.sha256(manifest_path.read_bytes()).hexdigest(),
    "inclusion_review_sha256": hashlib.sha256(review_path.read_bytes()).hexdigest(),
    "notebook_code_sha256": hashlib.sha256(code_source.encode()).hexdigest(),
    "requirements_sha256": hashlib.sha256((project_root / "requirements.txt").read_bytes()).hexdigest(),
    "rows": len(metadata_rows),
    "status": "first publication batch curated; user spot-check pending",
    "paper_curation": li_2018_review,
    "missingness": missingness,
}
curation_version = hashlib.sha256(json.dumps(curation_record, sort_keys=True).encode()).hexdigest()[:12]
table_path = curation_root / f"fibril_metadata_{curation_version}.csv"
record_path = table_path.with_suffix(".json")
for path, content in (
    (table_path, table_bytes),
    (record_path, (json.dumps(curation_record, indent=2, sort_keys=True) + "\n").encode()),
):
    if path.exists():
        assert path.read_bytes() == content, f"existing version differs: {path}"
    else:
        pending_path = path.with_suffix(path.suffix + ".part")
        pending_path.write_bytes(content)
        pending_path.replace(path)
print(f"{len(metadata_rows)} entries saved to {table_path.relative_to(project_root)}")

170 entries saved to data/curated/fibril_metadata_99f3303c9842.csv


show field coverage, the metadata preview and the first publication batch for spot-checking

In [7]:
def display_fibril_metadata(rows, columns):
    header = "".join(f"<th>{escape(column.replace('_', ' '))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row[column]))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


display_fibril_metadata(missingness, ["field", "reported", "unknown", "distinct_reported"])
display_fibril_metadata(
    metadata_rows[:10],
    ["pdb_id", "study_doi", "deposited_source", "resolution_angstrom", "specimen_ph", "vitrification_temperature_k"],
)

display_fibril_metadata(
    [
        {"field": field, "value": value if value is not None else "unknown",
         "raw": raw, "location": f"{document}: {location}",
         "note": li_2018_unknown_reasons.get(field, "")}
        for field, (value, raw, document, location) in li_2018_values.items()
    ],
    ["field", "value", "raw", "location", "note"],
)

field,reported,unknown,distinct reported
study_doi,170,0,58
species,170,0,1
deposited_source,169,1,3
source_notes,170,0,159
deposited_sequences,170,0,25
fragment_annotations,1,169,1
mutation_annotations,12,158,6
nonstandard_monomers,22,148,4
reported_authors,170,0,60
release_date,170,0,61


pdb id,study doi,deposited source,resolution angstrom,specimen ph,vitrification temperature k
10XU,10.1038/s41467-025-66146-4,"[""RECOMBINANT""]",3.18,7.4,unknown
12AP,10.64898/2026.04.07.716933,"[""NATURAL""]",2.1,7.4,unknown
12AQ,10.64898/2026.04.07.716933,"[""NATURAL""]",2.1,7.4,unknown
12AR,10.64898/2026.04.07.716933,"[""NATURAL""]",2.9,7.4,unknown
12AT,10.64898/2026.04.07.716933,"[""NATURAL""]",3.1,7.4,unknown
32DS,10.7554/elife.112692.1,"[""RECOMBINANT""]",2.6,7.0,unknown
32ED,10.7554/elife.112692.1,"[""RECOMBINANT""]",2.86,7.0,unknown
32EP,10.7554/elife.112692.1,"[""RECOMBINANT""]",2.8,7.0,unknown
32ER,10.7554/elife.112692.1,unknown,2.66,7.0,unknown
32HY,10.7554/elife.112692.1,"[""RECOMBINANT""]",2.92,7.0,unknown


field,value,raw,location,note
clinical_source,in_vitro,recombinant human aSyn,"main: pp.1-2, abstract and results",
construct_boundaries,human alpha-synuclein 1-140; full-length,1–140,"main: p.2, results; p.7, expression and purification",
growth_buffer_salt,15 mM tetrabutylphosphonium bromide; base buffer unspecified,15mM tetrabutylphosphonium bromide,"supplement: p.3, supplementary figure 2 caption",
growth_temperature_k,unknown,room temperature,"supplement: p.3, supplementary figure 2 caption",room temperature reported without a numeric value; no kelvin value inferred
growth_ph,unknown,None,"main: p.8, fibril preparation; selected condition",selected-preparation ph not stated; screening and specimen ph values are not assigned to growth
seeding_status,unknown,None,"main: p.8, fibril preparation; structural sample",seed addition not explicitly stated for the structural preparation; later seeding assays are downstream experiments
research_group,"UCLA collaboration: Jiang, Zhou, Eisenberg","{'affiliation': 'UCLA', 'supervision': 'L.J., Z.H.Z., and D.S.E.'}","main: p.1, affiliations; p.9, author contributions",
protofilament_count,2,two intertwined protofilaments,"main: p.2, cryo-em structures of two polymorphs",
source_preparation_id,li_2018_cryoem_preparation,one preparation,"main: p.2, introduction; p.9, data availability linking both pdb entries",
